In [1]:
!pip install -q streamlit requests pandas trafilatura cloudscraper seaborn pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 14.2 MB/s eta 0:00:00


In [8]:
%%writefile app.py
import streamlit as st
import pandas as pd
import google.generativeai as genai
from PIL import Image
import json

# --- 1. API Configuration ---
API_KEY = "API"
genai.configure(api_key=API_KEY)

def get_best_model():
    try:
        available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]
        priority = ['models/gemini-1.5-flash', 'models/gemini-1.5-pro']
        for p in priority:
            if p in available_models: return genai.GenerativeModel(p)
        return genai.GenerativeModel(available_models[0]) if available_models else None
    except Exception as e:
        st.error(f"API Error: {e}")
        return None

model = get_best_model()

# --- 2. Session State Management (Pipeline Steps) ---
if 'master_df' not in st.session_state:
    st.session_state.master_df = pd.DataFrame()
if 'detected_fields' not in st.session_state:
    st.session_state.detected_fields = []
if 'selected_fields' not in st.session_state:
    st.session_state.selected_fields = []
if 'temp_data' not in st.session_state:
    st.session_state.temp_data = None

st.set_page_config(page_title="Pro Form Digitizer", layout="wide")

st.title("📑 Pro-Level Form Digitizer")
st.markdown("A 4-step structured pipeline for precision data collection.")

# --- 3. Sidebar & Control Panel ---
with st.sidebar:
    if model:
        st.success(f"✅ Model: {model.model_name}")

    st.divider()
    st.subheader("🛠️ Control Panel")

    # BUTTON 1: Reset Current Workflow
    # This clears the current image's detection results but keeps the Master Database intact.
    if st.button("🔄 Reset Current Form", use_container_width=True, help="Clears detected fields and extraction results for the current image."):
        # Clear only process-related variables
        for key in ['detected_fields', 'selected_fields', 'temp_data']:
            st.session_state[key] = [] if 'fields' in key else None
        st.toast("Current process has been reset.")
        st.rerun()

    st.markdown("---")

    # BUTTON 2: Clear Master Database
    # Wrapped in an expander to prevent accidental deletion of saved data.
    with st.expander("🗑️ Delete All Records"):
        st.warning("Warning: This cannot be undone.")
        if st.button("🗑️ Clear All Saved Data", type="primary", use_container_width=True):
            st.session_state.master_df = pd.DataFrame()
            st.toast("Master Database cleared!")
            st.rerun()

# --- 4. Step 1 & 2: Image Upload & Schema Detection ---
img_file = st.file_uploader("Upload Form Image", type=['jpg', 'jpeg', 'png'])

if img_file:
    img = Image.open(img_file)
    if img.size[0] > 2000: img.thumbnail((1600, 1600))
    st.image(img, caption="Target Form", width=400)

    # STEP 1: Detect All Fields
    if st.button("🔍 Step 1: Detect All Available Fields"):
        with st.spinner('AI is identifying form schema...'):
            prompt = "List all the field names/labels present in this form as a simple list. Return ONLY a JSON list of strings."
            try:
                response = model.generate_content([prompt, img])
                res_text = response.text.strip().replace('```json', '').replace('```', '').strip()
                st.session_state.detected_fields = json.loads(res_text)
                st.session_state.temp_data = None # Reset subsequent steps
            except Exception as e:
                st.error("Failed to detect schema.")

    # STEP 2: Filter Fields
    if st.session_state.detected_fields:
        st.divider()
        st.subheader("🎯 Step 2: Select Fields to Extract")

        # Select All / Select None logic
        col1, col2 = st.columns(2)
        if col1.button("Select All"): st.session_state.selected_fields = st.session_state.detected_fields
        if col2.button("Clear Selection"): st.session_state.selected_fields = []

        st.session_state.selected_fields = st.multiselect(
            "Choose fields of interest:",
            options=st.session_state.detected_fields,
            default=st.session_state.selected_fields
        )

        # STEP 3: Extract Data
        if st.session_state.selected_fields:
            if st.button("🚀 Step 3: Extract Data for Selected Fields"):
                with st.spinner('Extracting specific handwritten values...'):
                    prompt = f"Extract handwritten values for ONLY these fields: {st.session_state.selected_fields}. Return as JSON."
                    try:
                        response = model.generate_content([prompt, img])
                        res_text = response.text.strip().replace('```json', '').replace('```', '').strip()
                        st.session_state.temp_data = json.loads(res_text)
                    except Exception as e:
                        st.error("Extraction failed.")

# --- 5. Step 4: Verify & Append ---
if st.session_state.temp_data:
    st.divider()
    st.subheader("📝 Step 4: Verify & Commit")

    # Render as an editable dataframe
    verify_df = pd.DataFrame([st.session_state.temp_data])
    edited_df = st.data_editor(verify_df, use_container_width=True, key="verify_editor")

    if st.button("✅ Data looks correct, save to Master", type="primary"):
        st.session_state.master_df = pd.concat(
            [st.session_state.master_df, edited_df],
            axis=0, ignore_index=True, sort=False
        )
        # Success and cleanup for next form
        st.session_state.temp_data = None
        st.success("Record saved to Master Database!")
        st.rerun()

# --- 6. Master Database View ---
st.divider()
st.subheader("📊 Master Database")
if not st.session_state.master_df.empty:
    st.dataframe(st.session_state.master_df, use_container_width=True)
    csv = st.session_state.master_df.to_csv(index=False).encode('utf-8-sig')
    st.download_button("📥 Download Combined CSV", data=csv, file_name="field_collected_data.csv")
else:
    st.caption("No records saved yet.")

Overwriting app.py


In [10]:
# 2. Set Auth Token
from pyngrok import ngrok
ngrok.set_auth_token("API2")
# Start Streamlit server on a specific port
!nohup streamlit run app.py --server.port 5011 &

# Start ngrok tunnel to expose the Streamlit server
ngrok_tunnel = ngrok.connect(addr='5011', proto='http', bind_tls=True)

# Print the URL of the ngrok tunnel
print(' * Tunnel URL:', ngrok_tunnel.public_url)

nohup: appending output to 'nohup.out'
 * Tunnel URL: https://c688-35-234-13-23.ngrok-free.app


In [9]:
# 1. Disconnect all active tunnels
try:
    tunnels = ngrok.get_tunnels()
    for tunnel in tunnels:
        print(f"Disconnecting: {tunnel.public_url}")
        ngrok.disconnect(tunnel.public_url)
except:
    pass

# 2. Terminate all ngrok and streamlit background processes
!pkill ngrok
!pkill streamlit

Disconnecting: https://c898-35-234-13-23.ngrok-free.app
